# Quét `image_size` trên T4 — mIoU vs ms/frame

**Mục đích:** trả lời *"cùng SAM 2, chạy `image_size` lớn trên GPU có chấm dứt được lỗi phân đoạn
HỤT diện tích (mIoU 0,7644 @192 trên CPU) không?"* — trước khi quyết định đổi lõi sang Cutie.

**Cần:** Runtime → Change runtime type → **T4 GPU**.

**Thời gian:** ô 2 ~3–6 phút (chỉ lần đầu) · ô 3 ~1 phút · ô 4 ~4 phút · ô 5 ~6 phút.

**Nguyên tắc số đo (AGENTS §6):** ô chọn suy từ GT frame 0, cùng bộ frame GT với lần đo CPU,
artifact có `git_rev` + `lenh`. `mIoU` là GT NGƯỜI TÔ, không phải cận trên tổng hợp.


In [ ]:
# 1) Kiểm GPU — phải thấy Tesla T4
!nvidia-smi
import torch
print("torch", torch.__version__, "| cuda", torch.cuda.is_available(),
      "|", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "KHÔNG CÓ GPU")
assert torch.cuda.is_available(), "Chưa bật GPU: Runtime → Change runtime type → T4 GPU"


In [ ]:
# 2) Lấy code + SAMURAI + checkpoint  (~3-6 phút, chỉ lần đầu; chạy lại không hỏng)
import os, subprocess
BRANCH = "colab-bench"

os.chdir("/content")
if not os.path.isdir("DoAn-NAB/.git"):
    subprocess.run(["git", "clone", "-q",
                    "https://github.com/BinhNguyen1912/DoAn-NAB.git"], check=True)
os.chdir("/content/DoAn-NAB")
subprocess.run(["git", "fetch", "-q", "origin"], check=False)
# `checkout -B <nhanh> origin/<nhanh>` = luôn khớp remote. KHÔNG dùng `git pull`: nhánh này bị
# force-push nên pull báo "divergent branches" (gặp thật 22-11) và cell chết giữa đường.
subprocess.run(["git", "checkout", "-q", "-B", BRANCH, f"origin/{BRANCH}"], check=False)
print("HEAD:", subprocess.run(["git", "log", "-1", "--oneline"],
                               capture_output=True, text=True).stdout.strip())

if not os.path.isdir("samurai/sam2"):
    subprocess.run(["git", "clone", "-q", "--depth", "1",
                    "https://github.com/yangchris11/samurai.git", "samurai"], check=True)
os.makedirs("samurai/sam2/checkpoints", exist_ok=True)
from scripts.colab_run_all import ensure_checkpoint   # có fallback CA + HuggingFace
_ckpt = "samurai/sam2/checkpoints/sam2.1_hiera_tiny.pt"
ensure_checkpoint(_ckpt)

# Colab ĐÃ có numpy/opencv/torch — KHÔNG cài lại (pip sẽ đòi restart runtime).
!pip install -q pyyaml loguru scipy
!pip install -q -e samurai/sam2
_size = os.path.getsize(_ckpt)
assert _size > 100_000_000, f"checkpoint tải HỎNG: {_size} bytes — chạy lại cell này"
print(f"cài đặt xong · checkpoint OK {_size/1e6:.1f} MB")


In [ ]:
# 3) Nạp clip binh2.mp4 — repo KHÔNG chứa video (`data/` bị .gitignore): phải nạp tay 1 lần/phiên
import os
os.chdir("/content/DoAn-NAB")
os.makedirs("data", exist_ok=True)
if not os.path.isfile("data/binh2.mp4"):
    from google.colab import files
    print("Chọn file binh2.mp4 từ máy bạn (3 MB):")
    up = files.upload()
    os.replace(next(iter(up)), "data/binh2.mp4")
print("clip OK:", os.path.getsize("data/binh2.mp4") // 1024, "KB")
for d in ("docs/gt/binh2_tay", "docs/gt/binh2_tay2"):
    assert os.path.isfile(f"{d}/index.json"), f"thiếu {d}/index.json — đang ở nhánh cũ?"
print("GT OK: binh2_tay + binh2_tay2")


In [ ]:
# 4) Quét 192 + 320 (nhẹ nhất). `--stream` in log TỪNG DÒNG để thấy nó đang chạy.
# Kỳ vọng: ~60-90 s "nạp model" rồi "propagate in video: ...%" rồi bảng số. TỔNG ~4 phút.
!cd /content/DoAn-NAB && python -u scripts/bench_gpu_binh2.py --device cuda --sizes 192,320 --stream


In [ ]:
# 5) Quét 512 + 1024 (nặng). Nếu báo `CUDA out of memory` thì bỏ 1024 — 512 là đủ để kết luận.
# Số của ô 4 được GIỮ LẠI (script gộp theo image_size), không bị ghi đè.
!cd /content/DoAn-NAB && python -u scripts/bench_gpu_binh2.py --device cuda --sizes 512,1024 --stream


In [ ]:
# 6) Bảng tổng hợp + tải artifact về máy
import glob, json, os
os.chdir("/content/DoAn-NAB")
p = sorted(glob.glob("data/gpu_sweep_binh2_*_metrics.json"))[-1]
d = json.load(open(p, encoding="utf-8"))
print("artifact:", p, "| git_rev:", d.get("git_rev"), "| dirty:", d.get("git_dirty_code"))
print("ô chọn (x,y,w,h):", d.get("o_chon_xywh"), "| frames GT:", len(d.get("frames_gt") or []))
print()
print(f"{'imgsz':>6} {'hieu luc':>9} {'ms/frame':>9} {'binh2_tay':>10} {'binh2_tay2':>11}")
for r in sorted(d["ket_qua"], key=lambda r: r.get("image_size") or 0):
    if "ma_thoat" in r:
        print(f"{r['image_size']:>6}  LỖI: {str(r.get('status_message'))[:90]}")
        continue
    m = r.get("mIoU") or {}
    print(f"{r['image_size']:>6} {str(r.get('image_size_hieu_luc')):>9} "
          f"{str(r.get('ms_moi_frame')):>9} {str(m.get('binh2_tay')):>10} {str(m.get('binh2_tay2')):>11}")
print()
print("Mốc so sánh — CPU laptop @192: mIoU 0,7644 · 342 ms/frame")
from google.colab import files
files.download(p)
